# 02. Basic Parsing and Guardrails

**Purpose:** This is the core **Prompt Engineering** notebook. It teaches baseline prompting, prompt tuning, and guardrails, addressing **Task 1** and the guardrails component of **Task 2**.

**Student Experience:**
*   **Act I – The Baseline:** You will run a starter prompt and obtain a baseline accuracy score.
*   **Act II – The Power of Prompts:** You will see how a deliberately "bad" prompt performs, then your main task is to edit and improve the prompt to outperform the baseline.
*   **Act III – Making It Robust:** You will encounter “garbage” inputs (e.g., “what is the weather?”) and learn to enable the `use_guardrails` flag to filter out invalid data.

**Key Takeaway:** You will discover that the quality of your instructions (the prompt) matters greatly. You will also learn the importance of guarding systems against invalid inputs.

## Step 1: Install Dependencies

In [ ]:
%pip install -q -r ../requirements.txt

## Step 2: Setup

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# Resolve repository root both for local runs and Google Colab.
PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents, Path('/content/esmt-workshop')]:
    if (candidate / 'src' / 'esmt_workshop').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Project root not found. Run this notebook from the ESMT_Workshop repository.'
    )

# Make local package importable inside notebook execution context.
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from esmt_workshop.student_api import process_batch_addresses
from esmt_workshop.evaluation import evaluate_predictions

print('PROJECT_ROOT =', PROJECT_ROOT)

from google.colab import auth
auth.authenticate_user()
gcloud_email = !gcloud config get-value account
STUDENT_EMAIL = gcloud_email[0] if gcloud_email else ""
os.environ['WORKSHOP_EMAIL'] = STUDENT_EMAIL
print(f"✅ Authenticated as: {STUDENT_EMAIL or 'Unknown'}")

## Step 3: Configuration and Data

In [ ]:
# We will use the fast and cheap model for this exercise.
MODEL_NAME = os.getenv('WORKSHOP_BASELINE_MODEL', 'gemini-2.5-flash')
TEMPERATURE = 0.0

# Load the development data
dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')

# Create a small sample for quick experiments
dev_sample = dev_df.head(20).copy()

# Create a version of the data with "garbage" inputs to test guardrails
garbage_data = pd.DataFrame([
    {'record_id': 'g1', 'Unstructured Address': 'book stores near me', 'Town Name': '', 'Postal Code': '', 'Country Code (2 characters)': ''},
    {'record_id': 'g2', 'Unstructured Address': 'what is the weather today?', 'Town Name': '', 'Postal Code': '', 'Country Code (2 characters)': ''},
])
dev_df_with_garbage = pd.concat([dev_df, garbage_data], ignore_index=True)

## Step 4: Prompt Cell

In [ ]:
# This is a decent "starter" prompt.
STARTER_PROMPT_TEMPLATE = '''You are an address parser for AML compliance.

Input address:
{address}

Return strict JSON only using this schema:
{schema}

Rules:
1. Town Name is only city/town/locality.
2. Postal Code includes only postal token(s).
3. Remaining Address includes everything else.
4. Country Code is ISO alpha-2 uppercase.
5. Use empty strings when uncertain.
'''

In [ ]:
# This is an example of a low-quality, "zero-effort" prompt.
BAD_PROMPT_TEMPLATE = "Parse this address: {address}. Give me JSON."

In [ ]:
# This is where you will write your improved prompt.
# Start by copying the STARTER_PROMPT_TEMPLATE and then try to improve it.
YOUR_CUSTOM_PROMPT = '''You are an AML address parser.

Input address:
{address}

Return strict JSON only with schema:
{schema}

Rules:
- Extract Town Name, Postal Code, Remaining Address, Country Code (2 characters).
- Country Code must be ISO alpha-2 uppercase.
- Do not invent values.
- Use empty strings if unknown.
'''

## Step 5: Execute Pipeline on Limited Dataset

Let's run our prompts on the small sample dataset to get quick feedback.

In [ ]:
def run_experiment(input_df: pd.DataFrame, prompt_template: str, *, use_guardrails: bool = False):
    """A helper function to run a parsing experiment and return the results."""
    print(f"Running experiment with {len(input_df)} records...")
    pred_df = process_batch_addresses(
        input_df,
        email=STUDENT_EMAIL,
        stage='prompt_tuned',
        model=MODEL_NAME,
        temperature=TEMPERATURE,
        use_guardrails=use_guardrails,
        custom_prompt_template=prompt_template,
    )
    report = evaluate_predictions(pred_df, input_df)
    return pred_df, report

In [ ]:
# Act I: The Baseline
print("--- Running Starter Prompt (Baseline) ---")
pred_starter, report_starter = run_experiment(dev_sample, STARTER_PROMPT_TEMPLATE)

In [ ]:
# Act II: The Power of Prompts
print("\n--- Running Bad Prompt ---")
pred_bad, report_bad = run_experiment(dev_sample, BAD_PROMPT_TEMPLATE)

print("\n--- Running Your Custom Prompt ---")
pred_custom, report_custom = run_experiment(dev_sample, YOUR_CUSTOM_PROMPT)

## Step 6: Analyze the Results from the Limited Dataset

Compare the accuracy scores from your experiments.

In [ ]:
print(f"Starter Prompt Accuracy: {report_starter['summary'].get('micro_accuracy', 0.0):.4f}")
print(f"Bad Prompt Accuracy:     {report_bad['summary'].get('micro_accuracy', 0.0):.4f}")
print(f"Your Prompt Accuracy:    {report_custom['summary'].get('micro_accuracy', 0.0):.4f}")

In [ ]:
# To debug your prompt, look at the rows where your model made a mistake.
print("\nDisplaying mismatches from your custom prompt:")
display(report_custom['mismatches'].head(10))

## Step 7: Execute Pipeline on Complete Dataset

Now let's see how your prompt performs on the full development set, and test our guardrails.

In [ ]:
# Act III: Making it Robust
print("--- Running on full dataset WITHOUT guardrails ---")
pred_full_no_guard, report_full_no_guard = run_experiment(dev_df_with_garbage, YOUR_CUSTOM_PROMPT, use_guardrails=False)

In [ ]:
print("\n--- Running on full dataset WITH guardrails ---")
pred_full_with_guard, report_full_with_guard = run_experiment(dev_df_with_garbage, YOUR_CUSTOM_PROMPT, use_guardrails=True)

## Step 8: Analyze the Full Run

In [ ]:
print(f"Accuracy on full set (no guardrails): {report_full_no_guard['summary'].get('micro_accuracy', 0.0):.4f}")
print(f"Accuracy on full set (with guardrails): {report_full_with_guard['summary'].get('micro_accuracy', 0.0):.4f}")

In [ ]:
# Let's see which inputs the guardrail correctly rejected.
rejected_inputs = pred_full_with_guard[pred_full_with_guard['valid_input'] == False]
print(f"\nThe guardrail rejected {len(rejected_inputs)} inputs:")
display(rejected_inputs)

## Step 9: Publish to Leaderboard

In [ ]:
PUBLISH_TO_LEADERBOARD = False # Set to True to publish your score

if PUBLISH_TO_LEADERBOARD:
    final_accuracy = report_full_with_guard['summary'].get('micro_accuracy', 0.0)
    print(f"Publishing score of {final_accuracy:.4f} to the leaderboard...")
    # In a real scenario, this would be a function to post the score.
    print("Leaderboard updated.")